[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sergiovillanueva/Modelos_Fundacionales/blob/main/4_DINO.ipynb)

# DINO: representaciones visuales sin etiquetas

DINO aprende a describir imagenes sin que nadie le diga que hay en ellas. No clasifica ni
detecta: convierte cada trozo de imagen en un vector, y con esos vectores se pueden hacer
muchas cosas.

En este cuaderno usamos **DINOv3** (agosto de 2025) y vemos, por orden:

1. Que devuelve exactamente el modelo: token CLS, tokens de registro y un vector por parche
2. Similitud coseno entre imagenes completas
3. Correspondencias entre parches de dos fotos distintas
4. Visualizacion de los parches con PCA, y por que suele salir ruido
5. Deteccion de anomalias en una pieza industrial, sin un solo ejemplo de defecto

DINOv2 sigue disponible cambiando una linea, y el codigo funciona igual con los dos.


## Configuracion e Imports

In [ ]:
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import requests
from io import BytesIO
import warnings

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version: {torch.__version__}, usando: {device}")
print(f"transformers version: {transformers.__version__}")


## El modelo

DINOv3 mejora sobre todo en caracteristicas densas, que es justo lo que usamos aqui.


In [ ]:
# Que DINO usamos. Cambiar de modelo es cambiar una linea: ELEGIDO.
#
#   dinov3-s   facebook/dinov3-vits16-pretrain-lvd1689m    21 M parametros, parche 16
#   dinov3-b   facebook/dinov3-vitb16-pretrain-lvd1689m    86 M parametros, parche 16
#   dinov2-b   facebook/dinov2-base                        86 M parametros, parche 14
#
# Los pesos de DINOv3 son de Meta y estan restringidos: hay que aceptar las condiciones
# en su pagina del Hub y autenticarse. En Colab:
#
#     from huggingface_hub import notebook_login
#     notebook_login()
#
# Si no puedes o no quieres, el cuaderno funciona igual con DINOv2: el codigo esta escrito
# para las dos familias. La unica diferencia real es que DINOv3 añade tokens de registro.

MODELOS = {
    "dinov3-s": "facebook/dinov3-vits16-pretrain-lvd1689m",
    "dinov3-b": "facebook/dinov3-vitb16-pretrain-lvd1689m",
    "dinov2-b": "facebook/dinov2-base",
}

ELEGIDO = "dinov3-s"
RESPALDO = "dinov2-b"

try:
    model_name = MODELOS[ELEGIDO]
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()
except Exception as error:
    print(f"No se pudo cargar {MODELOS[ELEGIDO]}: {type(error).__name__}")
    print("Suele ser que faltan las condiciones aceptadas o el token. Seguimos con el respaldo.\n")
    model_name = MODELOS[RESPALDO]
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()

PARCHE = model.config.patch_size
# DINOv3 añade tokens de registro entre el CLS y los parches. DINOv2 no tiene.
REGISTROS = getattr(model.config, "num_register_tokens", 0) or 0
DIMENSION = model.config.hidden_size

print(f"Modelo: {model_name}")
print(f"  tamaño de parche: {PARCHE} x {PARCHE} pixeles")
print(f"  tokens de registro: {REGISTROS}")
print(f"  dimension del embedding: {DIMENSION}")


## Carga de Imagenes de Ejemplo

In [ ]:
# Cargar imagenes desde GitHub
url_cars = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/cars.jpg"
image_cars = Image.open(BytesIO(requests.get(url_cars).content)).convert("RGB")

url_person_dog = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/person_dog.jpg"
image_person_dog = Image.open(BytesIO(requests.get(url_person_dog).content)).convert("RGB")

url_dog = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/dog.jpg"
image_dog = Image.open(BytesIO(requests.get(url_dog).content)).convert("RGB")

url_dog2 = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/dog2.jpg"
image_dog2 = Image.open(BytesIO(requests.get(url_dog2).content)).convert("RGB")

url_fruits = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/fruits.jpg"
image_fruits = Image.open(BytesIO(requests.get(url_fruits).content)).convert("RGB")

url_fruits3 = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/fruits3.jpg"
image_fruits3 = Image.open(BytesIO(requests.get(url_fruits3).content)).convert("RGB")   

url_bananas = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/bananas.jpg"
image_bananas = Image.open(BytesIO(requests.get(url_bananas).content)).convert("RGB")

url_person_banana = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/person_banana.jpg"
image_person_banana = Image.open(BytesIO(requests.get(url_person_banana).content)).convert("RGB")

url_carpet_ok = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/carpet_ok.jpg"
image_carpet_ok = Image.open(BytesIO(requests.get(url_carpet_ok).content)).convert("RGB")

url_carpet_nok = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/carpet_nok.jpg"
image_carpet_nok = Image.open(BytesIO(requests.get(url_carpet_nok).content)).convert("RGB")

print("Imagenes cargadas")

## 1. Que devuelve el modelo

DINO parte la imagen en cuadraditos (parches) y devuelve un vector por cada uno, mas un
vector `CLS` que resume la imagen entera. DINOv3 añade ademas cuatro **tokens de registro**,
que son memoria interna del modelo y no corresponden a ninguna zona de la imagen.

Esto ultimo es importante y es la primera causa del ruido que salia antes: si al cortar los
tokens no se saltan los registros, toda la rejilla queda desplazada cuatro posiciones y la
imagen de colores se convierte en estatica.


In [ ]:
def preparar(imagen, lado_largo=768):
    """Redimensiona a un multiplo del tamaño de parche, sin deformar.

    Importa mas de lo que parece: si el lado no es multiplo del parche, el modelo
    descarta la franja sobrante y la rejilla que calculemos deja de cuadrar con los
    tokens que devuelve.
    """
    ancho, alto = imagen.size
    escala = lado_largo / max(ancho, alto)
    nuevo_ancho = max(PARCHE, round(ancho * escala / PARCHE) * PARCHE)
    nuevo_alto = max(PARCHE, round(alto * escala / PARCHE) * PARCHE)
    return imagen.resize((nuevo_ancho, nuevo_alto), Image.BICUBIC)


def caracteristicas(imagen, lado_largo=768):
    """Pasa la imagen por DINO y separa los tres tipos de token.

    Devuelve (cls, parches, rejilla, imagen_usada):
      cls      vector de la imagen entera
      parches  una fila por parche, (numero_de_parches, dimension)
      rejilla  (filas, columnas) en que se ordenan esos parches
    """
    lista = imagen if isinstance(imagen, (list, tuple)) else [imagen]
    preparadas = [preparar(img, lado_largo) for img in lista]

    salidas = []
    for img in preparadas:
        entradas = processor(images=img, return_tensors="pt",
                             do_resize=False, do_center_crop=False).to(device)
        with torch.inference_mode():
            resultado = model(**entradas)

        alto_px, ancho_px = entradas["pixel_values"].shape[-2:]
        filas, columnas = alto_px // PARCHE, ancho_px // PARCHE

        tokens = resultado.last_hidden_state[0]
        cls = tokens[0].float().cpu().numpy()
        # Aqui esta la clave con DINOv3: hay que saltar CLS y los tokens de registro.
        parches = tokens[1 + REGISTROS:].float().cpu().numpy()

        if parches.shape[0] != filas * columnas:
            raise ValueError(
                f"Esperaba {filas * columnas} parches y hay {parches.shape[0]}. "
                "Revisa REGISTROS o el tamaño de entrada."
            )

        salidas.append((cls, parches, (filas, columnas), img))

    return salidas[0] if len(salidas) == 1 else salidas


# Veamos que sale exactamente
cls, parches, rejilla, usada = caracteristicas(image_dog)

print(f"Imagen original: {image_dog.size[0]} x {image_dog.size[1]} px")
print(f"Imagen usada:    {usada.size[0]} x {usada.size[1]} px")
print(f"Rejilla de parches: {rejilla[0]} filas x {rejilla[1]} columnas = {rejilla[0] * rejilla[1]} parches")
print(f"Tokens totales: 1 (CLS) + {REGISTROS} (registros) + {rejilla[0] * rejilla[1]} (parches)")
print(f"Cada parche es un vector de {parches.shape[1]} numeros")
print(f"La imagen entera cabe en {parches.shape[0] * parches.shape[1]:,} numeros")


### La rejilla, dibujada

Cada celda es un parche y a cada una le corresponde un vector. Con parches de 16 px y una
imagen de 768 px de lado largo salen unos 2.300 vectores por imagen.


In [ ]:
def ver_rejilla(imagen, lado_largo=768, cada=4):
    """Dibuja sobre la imagen la rejilla de parches que ve el modelo."""
    _, _, (filas, columnas), usada = caracteristicas(imagen, lado_largo)
    ancho, alto = usada.size

    plt.figure(figsize=(9, 9 * alto / ancho))
    plt.imshow(usada)
    for i in range(0, columnas + 1, cada):
        plt.axvline(i * PARCHE, color="white", linewidth=0.5, alpha=0.7)
    for j in range(0, filas + 1, cada):
        plt.axhline(j * PARCHE, color="white", linewidth=0.5, alpha=0.7)
    plt.title(f"{filas} x {columnas} parches de {PARCHE} px (una linea cada {cada})")
    plt.axis("off")
    plt.show()


ver_rejilla(image_dog)


## 2. Parecido entre imagenes: el token CLS

El vector `CLS` resume la imagen entera. Comparando dos con la similitud coseno sale un
numero entre -1 y 1. Nadie le enseño que es un perro: lo que mide es si las dos imagenes
estan descritas de forma parecida.


In [ ]:
def coseno(a, b):
    """Similitud coseno entre dos vectores."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


def matriz_similitud(imagenes, nombres):
    """Compara todas las imagenes contra todas usando el vector CLS."""
    vectores = [caracteristicas(img)[0] for img in imagenes]
    n = len(vectores)
    matriz = np.array([[coseno(vectores[i], vectores[j]) for j in range(n)] for i in range(n)])

    figura, ejes = plt.subplots(1, 2, figsize=(15, 6),
                               gridspec_kw={"width_ratios": [1.1, 1]})

    ejes[0].imshow(np.hstack([np.array(img.resize((160, 160))) for img in imagenes]))
    ejes[0].set_title(" · ".join(nombres), fontsize=10)
    ejes[0].axis("off")

    mapa = ejes[1].imshow(matriz, cmap="viridis", vmin=matriz.min(), vmax=1)
    ejes[1].set_xticks(range(n), nombres, rotation=45, ha="right")
    ejes[1].set_yticks(range(n), nombres)
    for i in range(n):
        for j in range(n):
            ejes[1].text(j, i, f"{matriz[i, j]:.2f}", ha="center", va="center",
                         color="white" if matriz[i, j] < 0.8 else "black", fontsize=9)
    ejes[1].set_title("Similitud coseno entre imagenes (CLS)")
    plt.colorbar(mapa, ax=ejes[1], fraction=0.046)
    plt.tight_layout()
    plt.show()

    return matriz


_ = matriz_similitud(
    [image_dog, image_dog2, image_person_dog, image_cars, image_fruits],
    ["perro", "perro 2", "persona+perro", "coches", "frutas"],
)


## 3. Correspondencias entre parches

Aqui se ve mejor lo que hay dentro. Elegimos **un parche** de una imagen y buscamos que
parche de la otra se le parece mas. Si el modelo entiende de verdad lo que mira, la oreja de
un perro encuentra la oreja del otro perro, aunque la raza, la pose y el fondo sean distintos.

Es la misma similitud coseno del tema 3, pero aplicada region a region en lugar de imagen a
imagen.


In [ ]:
def correspondencias(imagen_a, imagen_b, punto_relativo=(0.5, 0.4), lado_largo=512):
    """Elige un parche de la primera imagen y busca su parecido en la segunda.

    punto_relativo va de 0 a 1, asi que (0.5, 0.4) es el centro un poco hacia arriba.
    """
    (_, parches_a, (fa, ca), usada_a), (_, parches_b, (fb, cb), usada_b) = caracteristicas(
        [imagen_a, imagen_b], lado_largo
    )

    columna = min(int(punto_relativo[0] * ca), ca - 1)
    fila = min(int(punto_relativo[1] * fa), fa - 1)
    consulta = parches_a[fila * ca + columna]

    normal_b = parches_b / (np.linalg.norm(parches_b, axis=1, keepdims=True) + 1e-9)
    normal_q = consulta / (np.linalg.norm(consulta) + 1e-9)
    mapa = (normal_b @ normal_q).reshape(fb, cb)

    mejor = np.unravel_index(mapa.argmax(), mapa.shape)

    figura, ejes = plt.subplots(1, 3, figsize=(16, 5))

    ejes[0].imshow(usada_a)
    ejes[0].add_patch(plt.Rectangle((columna * PARCHE, fila * PARCHE), PARCHE, PARCHE,
                                    edgecolor="yellow", facecolor="none", linewidth=3))
    ejes[0].set_title("Parche elegido")
    ejes[0].axis("off")

    ejes[1].imshow(usada_b)
    ejes[1].imshow(np.array(Image.fromarray((mapa * 255).astype(np.uint8)).resize(usada_b.size, Image.BICUBIC)),
                   cmap="inferno", alpha=0.6)
    ejes[1].add_patch(plt.Rectangle((mejor[1] * PARCHE, mejor[0] * PARCHE), PARCHE, PARCHE,
                                    edgecolor="cyan", facecolor="none", linewidth=3))
    ejes[1].set_title(f"Donde se parece mas (max {mapa.max():.2f})")
    ejes[1].axis("off")

    ejes[2].hist(mapa.ravel(), bins=40, color="#2727c7")
    ejes[2].set_title("Reparto de similitudes")
    ejes[2].set_xlabel("coseno")

    plt.tight_layout()
    plt.show()

    return mapa


# La cabeza del perro de una foto, buscada en la otra
_ = correspondencias(image_dog, image_dog2, punto_relativo=(0.5, 0.35))

# Prueba tambien con:
# _ = correspondencias(image_person_dog, image_dog, punto_relativo=(0.35, 0.6))
# _ = correspondencias(image_fruits, image_fruits3, punto_relativo=(0.5, 0.5))


## 4. Ver los parches con PCA, bien hecho

Cada parche es un vector de cientos de numeros. Para verlo, se reducen a tres con PCA y se
pintan como rojo, verde y azul. Dos parches con colores parecidos son dos parches que el
modelo describe parecido.

**Por que salia ruido.** Tres motivos, y los tres estan corregidos arriba:

1. **Los tokens de registro.** Con DINOv3 hay cuatro entre el CLS y los parches. Si no se
   saltan, la rejilla se descoloca entera.
2. **El tamaño de entrada.** Si el lado de la imagen no es multiplo del parche, el modelo
   descarta la franja sobrante y la rejilla deja de cuadrar con los tokens.
3. **El fondo.** Es la causa mas visible. La primera componente principal casi siempre
   separa objeto y fondo, asi que si se usan las tres componentes sobre toda la imagen, dos
   se gastan en el cielo y la hierba y queda poco para distinguir las partes del objeto.

La receta de las demos oficiales es: primera componente para hacer una mascara, y repetir el
PCA usando solo los parches del objeto. La celda siguiente enseña las dos versiones juntas.


In [ ]:
def mapa_pca(parches, rejilla, quitar_fondo=True, umbral=0.5):
    """Convierte los parches en una imagen de colores con PCA.

    El truco de las demos de DINO, en tres pasos:
      1. Normalizar los vectores, para que la escala no mande sobre la direccion.
      2. Primera componente para separar objeto y fondo, y quedarse con el objeto.
      3. Repetir el PCA solo con el objeto: asi las tres componentes se gastan en
         distinguir sus partes y no en separar el objeto del cielo.
    """
    filas, columnas = rejilla
    X = parches.astype(np.float32)
    X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)

    if quitar_fondo:
        pc1 = PCA(n_components=1).fit_transform(X)[:, 0]
        pc1 = (pc1 - pc1.min()) / (np.ptp(pc1) + 1e-9)
        primer_plano = pc1 > umbral
        # El signo de una componente principal es arbitrario. Nos quedamos con el lado
        # minoritario, que casi siempre es el objeto y no el fondo.
        if primer_plano.mean() > 0.5:
            primer_plano = ~primer_plano
    else:
        primer_plano = np.ones(len(X), dtype=bool)

    componentes = PCA(n_components=3).fit_transform(X[primer_plano])
    bajo = np.percentile(componentes, 2, axis=0)
    alto = np.percentile(componentes, 98, axis=0)
    componentes = np.clip((componentes - bajo) / (alto - bajo + 1e-9), 0, 1)

    salida = np.zeros((filas * columnas, 3), dtype=np.float32)
    salida[primer_plano] = componentes
    return salida.reshape(filas, columnas, 3), primer_plano.reshape(filas, columnas)


def comparar_pca(imagen, lado_largo=768):
    """Enseña el PCA ingenuo y el PCA con el fondo fuera, uno al lado del otro."""
    _, parches, rejilla, usada = caracteristicas(imagen, lado_largo)

    ingenuo, _ = mapa_pca(parches, rejilla, quitar_fondo=False)
    limpio, mascara = mapa_pca(parches, rejilla, quitar_fondo=True)

    figura, ejes = plt.subplots(1, 4, figsize=(19, 5))
    for eje, contenido, titulo in zip(
        ejes,
        [np.array(usada), ingenuo, mascara.astype(float), limpio],
        ["Imagen", "PCA sobre todo\n(asi salia antes)", "Primer plano segun\nla 1a componente",
         "PCA solo del objeto\n(como en las demos)"],
    ):
        eje.imshow(contenido, cmap="gray" if contenido.ndim == 2 else None,
                   interpolation="nearest")
        eje.set_title(titulo, fontsize=11)
        eje.axis("off")
    plt.tight_layout()
    plt.show()


comparar_pca(image_dog)
# comparar_pca(image_person_dog)
# comparar_pca(image_cars)


### El mismo PCA para varias imagenes

Ajustar el PCA por separado en cada imagen tiene una trampa: cada ajuste elige sus propias
direcciones, asi que el mismo color no significa lo mismo en dos imagenes. Ajustandolo una
sola vez con los parches de todas, los colores pasan a ser comparables y se ve que el modelo
coloca las mismas partes en el mismo sitio.


In [ ]:
def pca_compartido(imagenes, nombres, lado_largo=640, quitar_fondo=True):
    """Un solo PCA para varias imagenes: los colores pasan a ser comparables.

    Ajustado por separado, cada imagen elige sus propias direcciones y el mismo color
    significa cosas distintas en cada una.
    """
    datos = caracteristicas(list(imagenes), lado_largo)
    todos = np.concatenate([d[1] for d in datos])
    todos = todos / (np.linalg.norm(todos, axis=1, keepdims=True) + 1e-9)

    if quitar_fondo:
        pc1 = PCA(n_components=1).fit_transform(todos)[:, 0]
        pc1 = (pc1 - pc1.min()) / (np.ptp(pc1) + 1e-9)
        primer_plano = pc1 > 0.5
        if primer_plano.mean() > 0.5:
            primer_plano = ~primer_plano
    else:
        primer_plano = np.ones(len(todos), dtype=bool)

    pca = PCA(n_components=3).fit(todos[primer_plano])
    proyectado = pca.transform(todos)
    bajo = np.percentile(proyectado[primer_plano], 2, axis=0)
    alto = np.percentile(proyectado[primer_plano], 98, axis=0)
    proyectado = np.clip((proyectado - bajo) / (alto - bajo + 1e-9), 0, 1)
    proyectado[~primer_plano] = 0

    figura, ejes = plt.subplots(2, len(datos), figsize=(5 * len(datos), 9))
    inicio = 0
    for columna, (_, parches, (filas, cols), usada) in enumerate(datos):
        trozo = proyectado[inicio:inicio + filas * cols].reshape(filas, cols, 3)
        inicio += filas * cols
        ejes[0, columna].imshow(usada)
        ejes[0, columna].set_title(nombres[columna])
        ejes[0, columna].axis("off")
        ejes[1, columna].imshow(trozo, interpolation="nearest")
        ejes[1, columna].axis("off")
    plt.suptitle("Mismo PCA para todas: el color significa lo mismo en las cuatro", y=0.98)
    plt.tight_layout()
    plt.show()


pca_compartido(
    [image_dog, image_dog2, image_person_dog, image_cars],
    ["perro", "perro 2", "persona y perro", "coches"],
)


## 5. Anomalias: encontrar el defecto sin haber visto ninguno

Esta es la aplicacion industrial directa de todo lo anterior, y es la idea de **PatchCore**.

En una linea de produccion tienes miles de piezas buenas y casi ninguna mala, y las malas
fallan cada vez de una forma distinta. Entrenar un clasificador de defectos es inviable. Lo
que si se puede hacer:

1. Pasar unas cuantas piezas **buenas** por DINO y guardar sus parches. Eso es el banco de
   normalidad, y es todo el entrenamiento que hay.
2. Para una pieza nueva, medir cada parche contra el parche mas parecido del banco.
3. Si algun parche no se parece a nada de lo visto, ahi esta el defecto.

Usamos una alfombra buena y otra con un nudo de hilos sueltos, del material del curso.
Fijate en que el umbral se decide mirando la pieza buena, no la defectuosa: en la realidad no
tienes defectos de referencia.


In [ ]:
def banco_de_normalidad(imagenes_ok, lado_largo=640):
    """Guarda los parches de las piezas buenas. Es todo el entrenamiento que hay."""
    datos = caracteristicas(list(imagenes_ok), lado_largo)
    if not isinstance(datos, list):
        datos = [datos]
    banco = np.concatenate([d[1] for d in datos])
    banco = banco / (np.linalg.norm(banco, axis=1, keepdims=True) + 1e-9)
    print(f"Banco con {banco.shape[0]} parches de {len(datos)} imagen(es) buena(s)")
    return banco


def mapa_anomalia(banco, imagen, lado_largo=640, es_del_banco=False):
    """Distancia coseno de cada parche al parche bueno mas parecido.

    `es_del_banco=True` cuando la imagen que medimos es una de las que llenaron el banco.
    Entonces cada parche se encuentra a si mismo, la similitud sale 1 y la puntuacion 0,
    asi que hay que mirar al segundo mas parecido. Sin esto el umbral saldria cero y
    cualquier cosa se rechazaria.
    """
    _, parches, (filas, columnas), usada = caracteristicas(imagen, lado_largo)
    consulta = parches / (np.linalg.norm(parches, axis=1, keepdims=True) + 1e-9)

    # Producto escalar contra todo el banco: la similitud mas alta manda.
    similitud = consulta @ banco.T
    if es_del_banco and similitud.shape[1] > 1:
        mejor = np.partition(similitud, -2, axis=1)[:, -2]
    else:
        mejor = similitud.max(axis=1)

    puntuacion = 1.0 - mejor
    return puntuacion.reshape(filas, columnas), usada


def inspeccionar(banco, imagen, umbral, titulo="", es_del_banco=False):
    """Pinta el mapa de anomalia y decide si la pieza pasa."""
    mapa, usada = mapa_anomalia(banco, imagen, es_del_banco=es_del_banco)
    marcadas = int((mapa > umbral).sum())
    veredicto = "SE RECHAZA" if marcadas else "se acepta"

    grande = np.array(Image.fromarray((mapa / (mapa.max() + 1e-9) * 255).astype(np.uint8))
                      .resize(usada.size, Image.BICUBIC)) / 255.0

    figura, ejes = plt.subplots(1, 3, figsize=(16, 5))
    ejes[0].imshow(usada)
    ejes[0].set_title("Pieza")
    ejes[0].axis("off")

    ejes[1].imshow(usada)
    ejes[1].imshow(grande, cmap="inferno", alpha=0.55)
    ejes[1].set_title(f"Mapa de anomalia (max {mapa.max():.3f})")
    ejes[1].axis("off")

    ejes[2].imshow(usada)
    ejes[2].imshow((mapa > umbral).astype(float), cmap="Reds", alpha=0.5,
                   extent=(0, usada.size[0], usada.size[1], 0), interpolation="nearest")
    ejes[2].set_title(f"Por encima de {umbral:.3f}: {marcadas} parches")
    ejes[2].axis("off")

    plt.suptitle(f"{titulo} → {veredicto}", fontsize=14)
    plt.tight_layout()
    plt.show()

    return mapa


banco = banco_de_normalidad([image_carpet_ok])

# El umbral se fija mirando la pieza buena, nunca la defectuosa: en produccion no tienes
# defectos de referencia. Dejamos un margen por encima de lo peor que da una pieza correcta.
mapa_ok = mapa_anomalia(banco, image_carpet_ok, es_del_banco=True)[0]
umbral = float(np.percentile(mapa_ok, 99.5)) * 1.2
print(f"Puntuacion de la pieza buena: mediana {np.median(mapa_ok):.4f}, maxima {mapa_ok.max():.4f}")
print(f"Umbral elegido: {umbral:.4f}")

_ = inspeccionar(banco, image_carpet_ok, umbral, "Alfombra buena", es_del_banco=True)
_ = inspeccionar(banco, image_carpet_nok, umbral, "Alfombra con defecto")


## Ejercicios


In [ ]:
# EJERCICIO 1: mueve el umbral y mira que pasa
#
# Con el umbral muy bajo, la pieza buena tambien se rechaza (falsa alarma).
# Con el umbral muy alto, el defecto se cuela (escape).
#
# _ = inspeccionar(banco, image_carpet_ok, umbral * 0.4, "Alfombra buena, umbral bajo", es_del_banco=True)
# _ = inspeccionar(banco, image_carpet_nok, umbral * 2.0, "Alfombra con defecto, umbral alto")


# EJERCICIO 2: usa tus propias imagenes
#
# mi_ok = Image.open("pieza_buena.jpg").convert("RGB")
# mi_nok = Image.open("pieza_mala.jpg").convert("RGB")
# mi_banco = banco_de_normalidad([mi_ok])
# _ = inspeccionar(mi_banco, mi_nok, umbral)


# EJERCICIO 3: cambia de modelo
#
# Vuelve a la celda del modelo, pon ELEGIDO = "dinov2-b" y ejecuta el cuaderno otra vez.
# Compara los mapas de PCA y el maximo del mapa de anomalia. Con DINOv2 el margen entre
# la pieza buena y la defectuosa suele ser mas estrecho.


## Resumen

En este cuaderno hemos visto:

✅ Que es el aprendizaje auto-supervisado y que devuelve DINO
✅ Los tres tipos de token: CLS, registros y parches, y por que hay que separarlos bien
✅ La rejilla de parches dibujada sobre la imagen
✅ Similitud coseno entre imagenes con el vector CLS
✅ Correspondencias parche a parche entre dos fotos distintas
✅ PCA de los parches a color, y las tres razones por las que salia ruido
✅ Un solo PCA para varias imagenes, con colores comparables
✅ Deteccion de anomalias sin un solo ejemplo de defecto, con banco de normalidad y umbral

**La idea que se lleva todo:** DINO no clasifica ni detecta, solo describe. Cada parche se
convierte en un vector, y a partir de ahi todo son distancias: parecidos entre imagenes,
correspondencias entre regiones, colores por PCA y anomalias. Esa es la razon de llamarlo
modelo fundacional.
